In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/kermany2018")

print("Path to dataset files:", path)

In [ ]:
!ls '/root/.cache/kagglehub/datasets/paultimothymooney/kermany2018/versions/2/OCT2017 /train/NORMAL'

In [ ]:
from glob import glob

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from PIL import Image

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
class OCTNoisyDataset(Dataset):
  def __init__(self, folder, transform=None, noise_factor=0.4, limit=None):
    self.paths = sorted(glob(f'{folder}/*.jpeg'))

    if limit: self.paths = self.paths[:limit]

    self.transform = transform
    self.noise_factor = noise_factor

  def __len__(self):
    return len(self.paths)

  def __getitem__(self, idx):
    path = self.paths[idx]

    image = Image.open(path).convert('L')

    clean = self.transform(image)
    noise = self.noise_factor * torch.randn_like(clean)

    noisy = torch.clamp(clean+noise, 0., 1)

    return clean, noisy

In [ ]:
batch_size = 32
learning_rate = 0.001
num_epochs = 10

In [ ]:
transform = transforms.Compose([
  transforms.Resize((128, 128)),
  transforms.ToTensor(),
])

data_root = '/root/.cache/kagglehub/datasets/paultimothymooney/kermany2018/versions/2/OCT2017 '

train_dataset = OCTNoisyDataset(
  f'{data_root}/train/NORMAL',
  transform=transform,
  noise_factor=0.2,
  limit=2000,
)

val_dataset = OCTNoisyDataset(
  f'{data_root}/val/NORMAL',
  transform=transform,
  noise_factor=0.2,
  limit=400,
)

test_dataset = OCTNoisyDataset(
  f'{data_root}/test/NORMAL',
  transform=transform,
  noise_factor=0.2,
  limit=400,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class DenoisingAutoencoder(nn.Module):

  def __init__(self):
    super().__init__()

    self.encoder = nn.Sequential(nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
                                  nn.BatchNorm2d(32),
                                  nn.LeakyReLU(),

                                  nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
                                  nn.BatchNorm2d(64),
                                  nn.LeakyReLU(),

                                  nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
                                  nn.BatchNorm2d(128),
                                  nn.LeakyReLU(),

                                  nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
                                  nn.BatchNorm2d(256),
                                  nn.LeakyReLU()

                                  )

    self.decoder = nn.Sequential(nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
                                  nn.BatchNorm2d(128),
                                  nn.LeakyReLU(),

                                  nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
                                  nn.BatchNorm2d(64),
                                  nn.LeakyReLU(),

                                  nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
                                  nn.BatchNorm2d(32),
                                  nn.LeakyReLU(),

                                  nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=1),
                                  nn.Sigmoid()
                                  )

  def forward(self, x):
    encoded = self.encoder(x)
    decoded = self.decoder(encoded)

    return decoded

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = DenoisingAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
for epoch in range(num_epochs):
  model.train()

  train_loss = 0

  for clean, noisy in train_loader:
    noisy, clean = noisy.to(device), clean.to(device)

    optimizer.zero_grad()

    output = model(noisy)
    loss = criterion(output, clean)

    loss.backward()
    optimizer.step()

    train_loss += loss.item()

  train_loss /= len(train_loader)

  model.eval()
  val_loss=0

  with torch.no_grad():
    for noisy, clean in val_loader:
      noisy, clean = noisy.to(device), clean.to(device)

      output = model(noisy)
      loss = criterion(output, clean)

      val_loss += loss.item()
  val_loss /= len(val_loader)

  print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

In [ ]:
model.eval()
test_loss = 0

with torch.no_grad():
  for clean, noisy in test_loader:
    noisy, clean = noisy.to(device), clean.to(device)

    output = model(noisy)
    loss = criterion(output, clean)

    test_loss += loss.item()

  print(f'Test Loss: {test_loss/len(test_loader):.4f}')

In [ ]:
def visualize_result(model, dataset, index=0):

  model.eval()

  clean, noisy = dataset[index]
  noisy = noisy.unsqueeze(0).to(device)

  with torch.no_grad():
    output = model(noisy).squeeze().cpu()

  fig, axes = plt.subplots(1, 3, figsize=(12, 4))

  axes[0].imshow(noisy.squeeze().cpu(), cmap='gray')
  axes[0].set_title('Noisy Image')
  axes[0].axis('off')

  axes[1].imshow(clean.squeeze(), cmap='gray')
  axes[1].set_title('Original Image')
  axes[1].axis('off')

  axes[2].imshow(output.squeeze(), cmap='gray')
  axes[2].set_title('Denoised Image')
  axes[2].axis('off')

  plt.show()

In [ ]:
visualize_result(model, test_dataset, index=10)

In [ ]:
visualize_result(model, test_dataset, index=5)